# 01 — Generate Customer & Order Data

Uses Faker to generate:
- **~200 customers** with unique IDs and emails
- **~500 orders** referencing valid customer IDs
- **~30 new_customers_batch** with deliberate duplicates (used later to trigger constraint violations)

In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
from faker import Faker
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, FloatType, DateType
)
import os
import random

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "pk_unique_demo")

fake = Faker()
Faker.seed(42)
random.seed(42)

## Generate Customers

In [2]:
NUM_CUSTOMERS = 200

customers = []
used_emails = set()

for i in range(1, NUM_CUSTOMERS + 1):
    # Ensure unique emails
    email = fake.email()
    while email in used_emails:
        email = fake.email()
    used_emails.add(email)

    customers.append((
        i,
        email,
        fake.first_name(),
        fake.last_name(),
        fake.phone_number(),
        fake.city(),
        fake.state_abbr(),
        fake.date_between(start_date="-3y", end_date="today"),
    ))

customer_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("email", StringType(), False),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("signup_date", DateType(), True),
])

customers_df = spark.createDataFrame(customers, schema=customer_schema)
customers_df.write.mode("overwrite").saveAsTable(f"{UC_CATALOG}.{UC_SCHEMA}.customers")

print(f"Wrote {customers_df.count()} customers")
customers_df.show(5, truncate=False)

Wrote 200 customers
+-----------+-------------------------+----------+---------+---------------------+---------------+-----+-----------+
|customer_id|email                    |first_name|last_name|phone                |city           |state|signup_date|
+-----------+-------------------------+----------+---------+---------------------+---------------+-----+-----------+
|1          |johnsonjoshua@example.org|Donald    |Garcia   |001-581-896-0013x3890|South Bridget  |AS   |2025-10-30 |
|2          |helenpeterson@example.org|Jason     |Gallagher|916-315-5940         |Millerport     |MP   |2025-02-08 |
|3          |barbara10@example.net    |Jeremy    |Roberts  |731-564-7525         |Jacquelineland |PA   |2025-02-15 |
|4          |frankgray@example.net    |Hailey    |Robinson |903-405-6413x95376   |Donaldside     |ND   |2024-01-04 |
|5          |zhurst@example.com       |Jeffrey   |Bright   |671.201.2269x16697   |Lake Nicoleview|WI   |2025-02-22 |
+-----------+-------------------------+-----

## Generate Orders

In [3]:
NUM_ORDERS = 500

products = [
    "Laptop", "Keyboard", "Mouse", "Monitor", "Headphones",
    "Webcam", "USB Hub", "Desk Lamp", "Notebook", "Backpack",
]

orders = []
for i in range(1, NUM_ORDERS + 1):
    orders.append((
        i,
        random.randint(1, NUM_CUSTOMERS),
        random.choice(products),
        random.randint(1, 5),
        round(random.uniform(9.99, 499.99), 2),
        fake.date_between(start_date="-1y", end_date="today"),
    ))

order_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("product_name", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", FloatType(), True),
    StructField("order_date", DateType(), True),
])

orders_df = spark.createDataFrame(orders, schema=order_schema)
orders_df.write.mode("overwrite").saveAsTable(f"{UC_CATALOG}.{UC_SCHEMA}.orders")

print(f"Wrote {orders_df.count()} orders")
orders_df.show(5, truncate=False)

Wrote 500 orders
+--------+-----------+------------+--------+------+----------+
|order_id|customer_id|product_name|quantity|price |order_date|
+--------+-----------+------------+--------+------+----------+
|1       |164        |Keyboard    |1       |373.35|2025-09-27|
|2       |63         |Monitor     |2       |370.86|2025-12-10|
|3       |174        |Notebook    |1       |299.33|2026-01-03|
|4       |9          |Laptop      |1       |117.12|2025-07-09|
|5       |130        |Backpack    |1       |285.0 |2025-10-13|
+--------+-----------+------------+--------+------+----------+
only showing top 5 rows


## Generate New Customers Batch (with deliberate duplicates)

This batch has:
- **10 genuinely new** customers
- **10 with duplicate `customer_id`** (IDs that already exist in customers)
- **10 with duplicate `email`** (emails that already exist in customers)

In [4]:
# Collect existing data for creating deliberate duplicates
existing_customers = spark.sql(
    f"SELECT customer_id, email FROM {UC_CATALOG}.{UC_SCHEMA}.customers"
).collect()

existing_ids = [row.customer_id for row in existing_customers]
existing_emails = [row.email for row in existing_customers]

batch = []

# 10 genuinely new customers (IDs starting after max existing)
max_id = max(existing_ids)
for i in range(1, 11):
    new_email = fake.email()
    while new_email in used_emails:
        new_email = fake.email()
    used_emails.add(new_email)

    batch.append((
        max_id + i,
        new_email,
        fake.first_name(),
        fake.last_name(),
        fake.phone_number(),
        fake.city(),
        fake.state_abbr(),
        fake.date_between(start_date="-1y", end_date="today"),
    ))

# 10 with duplicate customer_id (pick random existing IDs, new emails)
dup_ids = random.sample(existing_ids, 10)
for dup_id in dup_ids:
    new_email = fake.email()
    while new_email in used_emails:
        new_email = fake.email()
    used_emails.add(new_email)

    batch.append((
        dup_id,  # duplicate customer_id!
        new_email,
        fake.first_name(),
        fake.last_name(),
        fake.phone_number(),
        fake.city(),
        fake.state_abbr(),
        fake.date_between(start_date="-1y", end_date="today"),
    ))

# 10 with duplicate email (new IDs, but emails from existing customers)
dup_emails = random.sample(existing_emails, 10)
for dup_email in dup_emails:
    max_id += 11  # ensure unique IDs for this group
    batch.append((
        max_id,
        dup_email,  # duplicate email!
        fake.first_name(),
        fake.last_name(),
        fake.phone_number(),
        fake.city(),
        fake.state_abbr(),
        fake.date_between(start_date="-1y", end_date="today"),
    ))

batch_df = spark.createDataFrame(batch, schema=customer_schema)
batch_df.write.mode("overwrite").saveAsTable(f"{UC_CATALOG}.{UC_SCHEMA}.new_customers_batch")

print(f"Wrote {batch_df.count()} rows to new_customers_batch")
batch_df.show(30, truncate=False)

Wrote 30 rows to new_customers_batch
+-----------+----------------------------+----------+----------+----------------------+-----------------+-----+-----------+
|customer_id|email                       |first_name|last_name |phone                 |city             |state|signup_date|
+-----------+----------------------------+----------+----------+----------------------+-----------------+-----+-----------+
|201        |mckeetodd@example.net       |Amanda    |Lopez     |778.700.3845x76205    |Joelhaven        |KY   |2026-01-13 |
|202        |natasha08@example.com       |Jeremiah  |Peterson  |(721)968-9363x9264    |East Denise      |OR   |2025-10-06 |
|203        |michelleparker@example.net  |Rachel    |Browning  |(829)825-1881x25205   |Scottview        |FL   |2026-01-01 |
|204        |andersonanthony@example.net |Christina |Obrien    |+1-939-733-0517x289   |Port Paul        |NJ   |2025-05-11 |
|205        |jeremyortiz@example.org     |Sara      |Clements  |933.512.7975          |Sullivan

## Summary

In [5]:
for table in ["customers", "orders", "new_customers_batch"]:
    count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {UC_CATALOG}.{UC_SCHEMA}.{table}").collect()[0].cnt
    print(f"{UC_CATALOG}.{UC_SCHEMA}.{table}: {count} rows")

alexander_booth.pk_unique_demo.customers: 200 rows
alexander_booth.pk_unique_demo.orders: 500 rows
alexander_booth.pk_unique_demo.new_customers_batch: 30 rows
